In [1]:
import pandas as pd
import numpy as np

In [2]:
games = pd.read_csv('../data/raw_games.csv')

In [3]:
games = games.dropna(subset=['home_score', 'away_score']).copy()
games['home_win'] = (games['home_score'] > games['away_score']).astype(int)
games['point_diff'] = games['home_score'] - games['away_score']

In [4]:
# Reshapes data into one row per team per game
home = games[[
    'game_id',
    'season',
    'week',
    'gameday',
    'home_team',
    'home_score',
    'away_score'
]].rename(columns={
        'home_team': 'team',
        'home_score': 'points_for',
        'away_score': 'points_against'
})
home['is_home'] = 1

away = games[[
    'game_id',
    'season',
    'week',
    'gameday',
    'away_team',
    'away_score',
    'home_score'
]].rename(columns={
        'away_team': 'team',
        'away_score': 'points_for',
        'home_score': 'points_against'
})
away['is_away'] = 0

team_games = pd.concat([home, away]).sort_values(['team', 'season', 'week'])
team_games['win'] = (team_games['points_for'] > team_games['points_against']).astype(int)


In [5]:
# Rolling averages

N = 5
team_games['roll_points_for'] = (
    team_games.groupby('team')['points_for']
    .transform(lambda x: x.shift(1).rolling(N, min_periods=1).mean())
)
team_games['roll_points_against'] = (
    team_games.groupby('team')['points_against']
    .transform(lambda x: x.shift(1).rolling(N, min_periods=1).mean())
)
team_games['roll_win_pct'] = (
    team_games.groupby('team')['win']
    .transform(lambda x: x.shift(1).rolling(N, min_periods=1).mean())
)

In [6]:
# Elo rating

elo = {team: 1500 for team in team_games['team'].unique()}
K = 20
elo_before = []

for _, row in games.sort_values(['season', 'week']).iterrows():
    h, a = row['home_team'], row['away_team']
    elo_before.append((row['game_id'], h, elo[h], a, elo[a]))
    
    expected_home = 1 / (1 + 10 ** ((elo[a] - elo[h]) / 400))
    actual_home = 1 if row['home_win'] == 1 else 0
    
    elo[h] += K * (actual_home - expected_home)
    elo[a] += K * ((1 - actual_home) - (1 - expected_home))
    
elo_df = pd.DataFrame(elo_before, columns = [
    'game_id',
    'home_team',
    'home_elo',
    'away_team',
    'away_elo'
])

In [7]:
# Merges rolling stats onto game table

home_feats = team_games[[
    'game_id',
    'team',
    'roll_points_for',
    'roll_points_against',
    'roll_win_pct'
]].rename(columns={
    'team': 'home_team',
    'roll_points_for': 'home_roll_pf',
    'roll_points_against': 'home_roll_pa',
    'roll_win_pct': 'home_roll_winpct'
})

away_feats = team_games[[
    'game_id',
    'team',
    'roll_points_for',
    'roll_points_against',
    'roll_win_pct'
]].rename(columns={
    'team': 'away_team',
    'roll_points_for': 'away_roll_pf',
    'roll_points_against': 'away_roll_pa',
    'roll_win_pct': 'away_roll_winpct'
})

model_ready = games.merge(home_feats, on=['game_id', 'home_team'], how='left')
model_ready = model_ready.merge(away_feats, on=['game_id', 'away_team'], how='left')
model_ready = model_ready.merge(elo_df[['game_id', 'home_elo', 'away_elo']], on='game_id', how='left')


model_ready.to_csv('../data/model_ready_games.csv', index=False)
print(model_ready.shape)

(3075, 56)
